# Session 2: Tool Use — AI Agents for Clinical Data — INSTRUCTOR VERSION

## Recap from Session 1

In Session 1, **you** decided the query sequence: find conditions, get patients, retrieve labs. **You** wrote each query (with Claude's help generating code) and chained results together.

Now the question: what if the LLM could decide the sequence itself? Instead of you telling it "first search conditions, then get patients," you just ask the clinical question and let the LLM figure out the steps.

That shift — from the LLM as a **code-writing assistant** to the LLM as a **decision-making agent** — is the core idea of this session.

## About the Patient Population

This FHIR server contains **1,027 synthetic patients** generated with clinically coupled phenotypes. The patients are distributed across **6 clinical groups**:

| # | Phenotype | Description |
|---|-----------|-------------|
| 1 | **Metabolic syndrome** | Elevated BMI, blood pressure, triglycerides, and fasting glucose — but no diabetes diagnosis yet. |
| 2 | **Early Type 2 diabetes** | Recently diagnosed Type 2 diabetes with mildly elevated HbA1c. Kidney function is normal. |
| 3 | **Type 2 diabetes with chronic kidney disease stage G2** | Established Type 2 diabetes with mildly reduced kidney function (eGFR 60–89). |
| 4 | **Advanced Type 2 diabetes with chronic kidney disease stage G3b** | Long-standing Type 2 diabetes with moderately-to-severely reduced kidney function (eGFR 30–44). Often on insulin and multiple medications. |
| 5 | **Type 1 diabetes with early nephropathy** | Type 1 diabetes (the body's immune system destroys insulin-producing cells) with early signs of kidney damage. Very low C-peptide. |
| 6 | **Type 1 diabetes with poor control and chronic kidney disease stage G3a** | Type 1 diabetes with poor glycemic control (high HbA1c) and moderately reduced kidney function (eGFR 45–59). |

These phenotypes are **clinically coupled** — patients with worse diabetes control tend to have worse kidney function, mirroring real-world patterns.

## Clinical Code Reference

### Diagnosis Codes (SNOMED CT)

| Code | Condition | Notes |
|------|-----------|-------|
| 44054006 | Type 2 diabetes mellitus | The body becomes resistant to insulin |
| 46635009 | Type 1 diabetes mellitus | The immune system destroys insulin-producing cells |
| 709044004 | Chronic kidney disease | Gradual loss of kidney function over time |

### Observation Codes (LOINC)

| Code | Test | What It Measures |
|------|------|-----------------|
| 4548-4 | Hemoglobin A1c (HbA1c) | Average blood sugar over 2–3 months |
| 2160-0 | Creatinine | Waste product filtered by kidneys |
| 33914-3 | Estimated glomerular filtration rate (eGFR) | How well kidneys filter blood |
| 14959-1 | Urine albumin/creatinine ratio (UACR) | Protein leakage indicating kidney damage |
| 1986-9 | C-peptide | Marker of insulin production by the pancreas |
| 85354-9 | Blood pressure panel | Systolic and diastolic blood pressure |
| 39156-5 | Body mass index (BMI) | Weight relative to height |
| 1558-6 | Fasting glucose | Blood sugar after overnight fasting |
| 2339-0 | Blood glucose | Random blood sugar measurement |
| 3094-0 | Blood urea nitrogen (BUN) | Another kidney function marker |
| 13457-7 | LDL cholesterol | "Bad" cholesterol |
| 2085-9 | HDL cholesterol | "Good" cholesterol |
| 2571-8 | Triglycerides | Blood fat linked to heart disease risk |

### Interpretation Thresholds

| Measure | Range | Interpretation |
|---------|-------|----------------|
| HbA1c | < 5.7% | Normal |
| | 5.7% – 6.4% | Prediabetes |
| | ≥ 6.5% | Diabetes |
| | **> 7.5%** | **Poor glycemic control** |
| eGFR | > 90 | Normal kidney function |
| | 60–89 | Mildly decreased |
| | 45–59 | Moderately decreased |
| | 30–44 | Moderately-to-severely decreased |
| | < 30 | Severely decreased |
| UACR | < 30 mg/g | Normal |
| | 30–300 mg/g | Moderately increased albuminuria |
| | > 300 mg/g | Severely increased albuminuria |

In [ ]:
!pip install anthropic requests pandas matplotlib

In [ ]:
import os, json, requests, urllib3
import pandas as pd
import matplotlib.pyplot as plt
from anthropic import Anthropic

# Suppress SSL warnings (self-signed cert on teaching server)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---- Anthropic Client ----
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except (ImportError, Exception):
    api_key = None

api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError(
        "Set ANTHROPIC_API_KEY in Colab Secrets (key icon in left sidebar) "
        "or as an environment variable."
    )

client = Anthropic(api_key=api_key)
MODEL = "claude-sonnet-4-20250514"

# ---- FHIR Server ----
FHIR_BASE = "https://lfh-fhir.eastus2.cloudapp.azure.com:9443/fhir-server/api/v4"
FHIR_SESSION = requests.Session()
FHIR_SESSION.auth = ("fhiruser", "BmI512@ccess")
FHIR_SESSION.verify = False

# Verify connections
resp = FHIR_SESSION.get(f"{FHIR_BASE}/metadata", params={"_format": "json"}, timeout=10)
if resp.status_code == 200:
    fhir_version = resp.json().get("fhirVersion", "unknown")
    print(f"\u2705 FHIR server connected (version {fhir_version})")
    count_resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Patient",
        params={"_summary": "count", "_format": "json"},
        timeout=10
    )
    if count_resp.status_code == 200:
        total = count_resp.json().get("total", "unknown")
        print(f"   {total} patients available")
else:
    print(f"\u274c FHIR server error: HTTP {resp.status_code}")

print(f"\u2705 Anthropic client ready (model: {MODEL})")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FHIR Tool Functions
# ══════════════════════════════════════════════════════════════

def search_conditions(code: str, max_results: int = 50) -> dict:
    """Search for conditions by SNOMED CT code."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Condition",
        params={"code": code, "_count": max_results, "_format": "json"},
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        results.append({
            "condition_id": r.get("id"),
            "code": coding.get("code", ""),
            "display": coding.get("display", ""),
            "patient_reference": r.get("subject", {}).get("reference", ""),
            "clinical_status": r.get("clinicalStatus", {}).get("coding", [{}])[0].get("code", ""),
        })
    return {"total": bundle.get("total"), "results": results}


def get_patient(patient_id: str) -> dict:
    """Get a single patient's demographics."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Patient/{patient_id}",
        params={"_format": "json"},
        timeout=30,
    )
    p = resp.json()
    name = p.get("name", [{}])[0]
    return {
        "id": p.get("id"),
        "name": f"{' '.join(name.get('given', []))} {name.get('family', '')}".strip(),
        "gender": p.get("gender", ""),
        "birthDate": p.get("birthDate", ""),
    }


def search_observations(patient_id: str, loinc_code: str, max_results: int = 5) -> dict:
    """Search for observations by patient and LOINC code."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Observation",
        params={
            "subject": f"Patient/{patient_id}",
            "code": loinc_code,
            "_count": max_results,
            "_sort": "-date",
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        value_qty = r.get("valueQuantity", {})
        result_entry = {
            "observation_id": r.get("id"),
            "code": r.get("code", {}).get("coding", [{}])[0].get("code", ""),
            "display": r.get("code", {}).get("coding", [{}])[0].get("display", ""),
            "value": value_qty.get("value"),
            "unit": value_qty.get("unit", ""),
            "date": r.get("effectiveDateTime", ""),
        }
        # Handle component-based observations (e.g., blood pressure)
        if not value_qty.get("value") and r.get("component"):
            components = []
            for comp in r["component"]:
                comp_code = comp.get("code", {}).get("coding", [{}])[0]
                comp_val = comp.get("valueQuantity", {})
                components.append({
                    "component": comp_code.get("display", ""),
                    "value": comp_val.get("value"),
                    "unit": comp_val.get("unit", ""),
                })
            result_entry["components"] = components
        results.append(result_entry)
    return {"total": bundle.get("total"), "results": results}


def search_medications(patient_id: str, max_results: int = 10) -> dict:
    """Search for medication requests for a patient."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/MedicationRequest",
        params={
            "subject": f"Patient/{patient_id}",
            "_count": max_results,
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        med_concept = r.get("medicationCodeableConcept", {})
        coding = med_concept.get("coding", [{}])[0] if med_concept.get("coding") else {}
        med_name = coding.get("display") or med_concept.get("text", "unknown")
        results.append({
            "medication": med_name,
            "code": coding.get("code", ""),
            "status": r.get("status", ""),
            "date": r.get("authoredOn", ""),
        })
    return {"total": bundle.get("total"), "results": results}


def search_all_conditions(patient_id: str, max_results: int = 20) -> dict:
    """Get all conditions (full problem list) for a specific patient."""
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Condition",
        params={
            "subject": f"Patient/{patient_id}",
            "_count": max_results,
            "_format": "json",
        },
        timeout=30,
    )
    bundle = resp.json()
    results = []
    for entry in bundle.get("entry", []):
        r = entry["resource"]
        coding = r.get("code", {}).get("coding", [{}])[0]
        results.append({
            "condition_id": r.get("id"),
            "code": coding.get("code", ""),
            "display": coding.get("display", ""),
            "clinical_status": r.get("clinicalStatus", {}).get("coding", [{}])[0].get("code", ""),
        })
    return {"total": bundle.get("total"), "results": results}


# ---- Smoke tests ----
print("Testing tool functions...")
_test = search_conditions("44054006", max_results=3)
print(f"  search_conditions('44054006'): {_test['total']} conditions found")
if _test["results"]:
    _pid = _test["results"][0]["patient_reference"].split("/")[-1]
    _p = get_patient(_pid)
    print(f"  get_patient('{_pid}'): {_p['name']}")
    _obs = search_observations(_pid, "4548-4", max_results=1)
    print(f"  search_observations(HbA1c): {_obs['total']} observations")
    _meds = search_medications(_pid)
    print(f"  search_medications: {_meds['total']} medications")
    _conds = search_all_conditions(_pid)
    print(f"  search_all_conditions: {_conds['total']} conditions")
print("\u2705 All tools working")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Tool Schemas — what the LLM sees
# ══════════════════════════════════════════════════════════════

tools = [
    {
        "name": "search_conditions",
        "description": (
            "Search for patient conditions by SNOMED CT diagnosis code. "
            "Returns matching conditions with patient references. "
            "Common codes: 44054006 (Type 2 diabetes mellitus), "
            "46635009 (Type 1 diabetes mellitus), "
            "709044004 (chronic kidney disease)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "SNOMED CT code (e.g., '44054006' for Type 2 diabetes)",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 50)",
                },
            },
            "required": ["code"],
        },
    },
    {
        "name": "get_patient",
        "description": (
            "Retrieve demographics for a single patient by ID. "
            "Returns name, gender, and birth date."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID (from a condition's patient_reference, e.g., 'abc123')",
                },
            },
            "required": ["patient_id"],
        },
    },
    {
        "name": "search_observations",
        "description": (
            "Search for clinical observations (lab results, vitals) for a patient "
            "by LOINC code. Returns values sorted by date (most recent first). "
            "Common codes: 4548-4 (HbA1c), 2160-0 (creatinine), 33914-3 (eGFR), "
            "1986-9 (C-peptide), 14959-1 (urine albumin/creatinine ratio), "
            "85354-9 (blood pressure), 39156-5 (BMI), 1558-6 (fasting glucose), "
            "2339-0 (blood glucose), 3094-0 (BUN), 13457-7 (LDL cholesterol), "
            "2085-9 (HDL cholesterol), 2571-8 (triglycerides)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "loinc_code": {
                    "type": "string",
                    "description": "LOINC code for the observation (e.g., '4548-4' for HbA1c)",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 5)",
                },
            },
            "required": ["patient_id", "loinc_code"],
        },
    },
    {
        "name": "search_medications",
        "description": (
            "Search for medication prescriptions (MedicationRequest resources) "
            "for a patient. Returns medication name, status, and date."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 10)",
                },
            },
            "required": ["patient_id"],
        },
    },
    {
        "name": "search_all_conditions",
        "description": (
            "Retrieve the complete problem list (all diagnoses) for a specific "
            "patient. Unlike search_conditions which finds patients by one "
            "diagnosis code, this returns ALL conditions for one patient."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "patient_id": {
                    "type": "string",
                    "description": "The patient ID",
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum results to return (default: 20)",
                },
            },
            "required": ["patient_id"],
        },
    },
]

available_functions = {
    "search_conditions": search_conditions,
    "get_patient": get_patient,
    "search_observations": search_observations,
    "search_medications": search_medications,
    "search_all_conditions": search_all_conditions,
}

## Understanding Tool Schemas

The LLM never sees your Python source code. It reads **JSON schemas** describing each tool: name, description, and parameters.

The quality of these descriptions directly affects agent performance. Notice the embedded LOINC and SNOMED codes in the descriptions — the LLM uses these to choose the right arguments.

There are **5 tools** available:

| Tool | Purpose |
|------|---------|
| `search_conditions` | Find patients by diagnosis code |
| `get_patient` | Get one patient's demographics |
| `search_observations` | Get lab results by LOINC code |
| `search_medications` | Get medication list |
| `search_all_conditions` | Get full problem list for one patient |

## Predict the Agent's Behavior

The agent will be asked:

> "Find patients with Type 2 diabetes (SNOMED code 44054006). For each patient, retrieve their most recent HbA1c, eGFR, and creatinine values, and list their current medications. Which patients have both poor glycemic control (HbA1c > 7.5%) and impaired kidney function (eGFR < 60)? What medications are they taking?"

**What tools will it use first?**

`search_conditions("44054006")` to find Type 2 diabetes patients, then `get_patient()` for demographics, then `search_observations()` for HbA1c (4548-4), eGFR (33914-3), creatinine (2160-0), then `search_medications()` for each patient.

**How many tool calls do you expect?**

~20-25 calls: 1 condition search + N×get_patient + N×3 observation searches + N×medication searches. May approach max_steps=25.

**What LOINC/SNOMED codes should it use?**

44054006 (Type 2 diabetes), 4548-4 (HbA1c), 33914-3 (eGFR), 2160-0 (creatinine)

**What could go wrong?**

Agent might not check all 3 lab types for every patient. Could run out of steps. Might not combine the data correctly in its summary.

In [ ]:

SYSTEM_PROMPT = \"\"\"You are a clinical data assistant with access to a FHIR \
server containing synthetic patient records.

PATIENT POPULATION: The server has approximately 1,027 patients across 6 \
clinical phenotypes:
1. Metabolic syndrome
2. Early Type 2 diabetes
3. Type 2 diabetes with chronic kidney disease stage G2
4. Advanced Type 2 diabetes with chronic kidney disease stage G3b
5. Type 1 diabetes with early nephropathy
6. Type 1 diabetes with poor control and chronic kidney disease stage G3a

DIAGNOSIS CODES (SNOMED CT):
- 44054006: Type 2 diabetes mellitus
- 46635009: Type 1 diabetes mellitus
- 709044004: Chronic kidney disease

OBSERVATION CODES (LOINC):
- 4548-4: Hemoglobin A1c (HbA1c) — glycemic control marker
- 2160-0: Creatinine — kidney function marker
- 33914-3: Estimated glomerular filtration rate (eGFR) — kidney function
- 14959-1: Urine albumin/creatinine ratio (UACR) — kidney damage marker
- 1986-9: C-peptide — insulin production marker (low in Type 1, normal/high in Type 2)
- 85354-9: Blood pressure panel
- 39156-5: Body mass index (BMI)
- 1558-6: Fasting glucose
- 2339-0: Blood glucose
- 3094-0: Blood urea nitrogen (BUN)
- 13457-7: LDL cholesterol
- 2085-9: HDL cholesterol
- 2571-8: Triglycerides

INTERPRETATION THRESHOLDS:
- HbA1c > 7.5%: poor glycemic control
- eGFR < 60 mL/min/1.73m²: impaired kidney function
- eGFR < 30: severely impaired kidney function
- UACR > 30 mg/g: moderately increased albuminuria
- UACR > 300 mg/g: severely increased albuminuria

STRATEGY — think step by step:
1. Identify the condition(s) relevant to the question using search_conditions
2. Extract patient references from the condition results
3. Get patient demographics with get_patient
4. Look up relevant observations using the appropriate LOINC codes
5. Check medications if relevant to the question
6. Synthesize findings into a clear clinical summary

RULES:
- NEVER invent or hallucinate data. Only report values actually returned by tools.
- If a tool returns no results for a patient, state that explicitly.
- Always identify patients by name when available.
- Show actual lab values, not just categories.
- When comparing groups, provide counts and specific values.
- Be thorough: check all relevant patients, not just the first few.
\"\"\"


In [ ]:

def run_agent(question, system_prompt=SYSTEM_PROMPT, tools=tools,
              available_functions=available_functions, max_steps=25):
    \"\"\"Run the tool-use agent loop.\"\"\"
    print(f"\U0001f916 AGENT QUESTION: {question}\n")
    print("=" * 70)

    messages = [{"role": "user", "content": question}]
    tool_calls_log = []
    step = 0

    while step < max_steps:
        step += 1
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )

        # Check for tool use
        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

        if not tool_use_blocks:
            # Final text response
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            print(f"\n{'=' * 70}")
            print(f"\u2705 FINAL ANSWER (after {step} steps):\n")
            print(final_text)
            return final_text, tool_calls_log, messages

        # Serialize assistant content for message history
        assistant_content = []
        for block in response.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                assistant_content.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input,
                })
        messages.append({"role": "assistant", "content": assistant_content})

        # Execute each tool call
        tool_results = []
        for block in tool_use_blocks:
            fn_name = block.name
            fn_args = block.input

            args_str = ", ".join(f"{k}={v!r}" for k, v in fn_args.items())
            print(f"\U0001f527 Step {step}: {fn_name}({args_str})")

            tool_calls_log.append({
                "step": step,
                "function": fn_name,
                "arguments": fn_args,
            })

            try:
                result = available_functions[fn_name](**fn_args)
            except Exception as e:
                result = {"error": str(e)}

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result, default=str),
            })

        messages.append({"role": "user", "content": tool_results})

    print(f"\n\u26a0\ufe0f Reached max steps ({max_steps})")
    return "Agent reached step limit.", tool_calls_log, messages


## Run the Agent

The cell below sends the clinical question to the agent. Watch the tool call trace as it runs — each line shows a function call the LLM decided to make.

In [ ]:
user_question = """Find patients with Type 2 diabetes (SNOMED code 44054006). For each patient, retrieve their most recent HbA1c, eGFR, and creatinine values, and list their current medications. Which patients have both poor glycemic control (HbA1c > 7.5%) and impaired kidney function (eGFR < 60)? What medications are they taking?"""

answer, tool_calls_log, messages = run_agent(user_question)

In [ ]:

# ── Tool Call Trace ──────────────────────────────────────────
print("\U0001f4cb TOOL CALL SEQUENCE\n")
print(f"{'Step':<6} {'Function':<25} {'Key Arguments'}")
print("-" * 70)
for tc in tool_calls_log:
    args_summary = ", ".join(f"{k}={v}" for k, v in tc["arguments"].items())
    print(f"{tc['step']:<6} {tc['function']:<25} {args_summary}")

print(f"\nTotal tool calls: {len(tool_calls_log)}")
tools_used = set(tc["function"] for tc in tool_calls_log)
print(f"Unique tools used: {sorted(tools_used)}")


## Compare to Your Predictions

**Did the agent use the tools you expected?**

Typically yes — the agent follows a logical workflow: conditions → patients → observations → medications. It usually starts with `search_conditions` and systematically works through each patient.

**Were there any surprises?**

The agent may not check all 3 lab types for every patient if it starts running low on steps. It might prioritize HbA1c over creatinine/eGFR. Some runs will check medications last or skip them for some patients.

**Did the agent correctly identify patients with both flags?**

Usually yes, but verify against the visualization below. The agent's textual summary should match the scatter plot's red dots below the eGFR=60 line.

In [ ]:

# ── Full Conversation Anatomy ────────────────────────────────
print("FULL CONVERSATION STRUCTURE\n")
for i, msg in enumerate(messages):
    role = msg["role"]
    content = msg["content"]

    if isinstance(content, str):
        print(f"[{i}] {role}: {content[:120]}...")
    elif isinstance(content, list):
        for block in content:
            if isinstance(block, dict):
                btype = block.get("type", "?")
                if btype == "text":
                    print(f"[{i}] {role}/text: {block['text'][:120]}...")
                elif btype == "tool_use":
                    print(f"[{i}] {role}/tool_use: {block['name']}({block['input']})")
                elif btype == "tool_result":
                    preview = block["content"][:80] if isinstance(block["content"], str) else str(block["content"])[:80]
                    print(f"[{i}] {role}/tool_result: {preview}...")
            else:
                print(f"[{i}] {role}: {str(block)[:120]}...")
    print()


## Visualize the Data

Let's visualize the data the agent retrieved. We'll call the same tool functions the agent used, but collect the results into a DataFrame for plotting.

This is cleaner than trying to parse the agent's text output — and it demonstrates how tool functions can be used both by agents and by traditional code.

In [ ]:

# ══════════════════════════════════════════════════════════════
# VISUALIZATION: Re-query and Plot
# ══════════════════════════════════════════════════════════════
# We call the same tool functions the agent used, but collect the results
# into DataFrames for plotting.

# Step 1: Get Type 2 diabetes patients
t2d_conditions = search_conditions("44054006", max_results=50)
t2d_patient_ids = list(set(
    r["patient_reference"].split("/")[-1]
    for r in t2d_conditions["results"]
))
print(f"Type 2 diabetes patients found: {len(t2d_patient_ids)}")

# Step 2: Collect HbA1c, eGFR, and medications
rows = []
all_meds = []
for pid in t2d_patient_ids:
    patient = get_patient(pid)
    hba1c = search_observations(pid, "4548-4", max_results=1)
    egfr = search_observations(pid, "33914-3", max_results=1)
    meds = search_medications(pid, max_results=10)

    hba1c_val = hba1c["results"][0]["value"] if hba1c["results"] else None
    egfr_val = egfr["results"][0]["value"] if egfr["results"] else None

    rows.append({
        "patient_id": pid,
        "name": patient["name"],
        "hba1c": hba1c_val,
        "egfr": egfr_val,
    })

    for m in meds.get("results", []):
        all_meds.append({"patient_id": pid, "medication": m["medication"]})

df = pd.DataFrame(rows)
df["hba1c"] = pd.to_numeric(df["hba1c"], errors="coerce")
df["egfr"] = pd.to_numeric(df["egfr"], errors="coerce")

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: HbA1c vs eGFR scatter
plot_df = df.dropna(subset=["hba1c", "egfr"])
if len(plot_df) > 0:
    colors = ["tab:red" if h > 7.5 else "tab:green" for h in plot_df["hba1c"]]
    axes[0].scatter(plot_df["hba1c"], plot_df["egfr"], c=colors, alpha=0.7,
                    edgecolors="black", linewidth=0.5, s=80)
    axes[0].axhline(y=60, color="gray", linestyle="--", alpha=0.7)
    axes[0].axvline(x=7.5, color="gray", linestyle=":", alpha=0.7)
    axes[0].set_xlabel("HbA1c (%)")
    axes[0].set_ylabel("eGFR (mL/min/1.73m\u00b2)")
    axes[0].set_title("HbA1c vs eGFR \u2014 Type 2 Diabetes")
else:
    axes[0].text(0.5, 0.5, "No data", ha="center", va="center")

# Right: Medication frequency bar chart
df_meds = pd.DataFrame(all_meds)
if len(df_meds) > 0:
    med_counts = df_meds["medication"].value_counts()
    med_counts.plot(kind="barh", ax=axes[1], color="steelblue")
    axes[1].set_xlabel("Number of Patients")
    axes[1].set_title("Medication Frequency \u2014 Type 2 Diabetes")
    axes[1].invert_yaxis()
else:
    axes[1].text(0.5, 0.5, "No medication data", ha="center", va="center")

plt.tight_layout()
plt.show()

print(f"\nPatients with both HbA1c and eGFR data: {len(plot_df)}")
if len(plot_df) > 0:
    both = sum((plot_df["hba1c"] > 7.5) & (plot_df["egfr"] < 60))
    print(f"Poor control + impaired kidneys: {both}")
if len(df_meds) > 0:
    print(f"Unique medications: {df_meds['medication'].nunique()}")


## Session 2 Takeaways

1. **Tool-use agents** let the LLM decide which functions to call and in what order, based on the question.
2. **The same clinical workflow** you built manually in Session 1 can be automated — but the agent's approach may differ from yours.
3. **Visualization** makes relationships tangible: the scatter plot reveals whether the agent's data supports the expected correlation between diabetes control and kidney function.
4. **Medications** add clinical context: patients with both poor control and impaired kidneys may be on more aggressive regimens.

In **Session 3**, you'll ask your own questions, add more tools, and actively look for where the agent breaks.